In [ ]:
import sys
import os
import importlib.util
import cv2
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import warnings
from skimage import io, transform
from skimage.util import img_as_ubyte


compressdiff_path = os.path.abspath("../defenselib/recompress_segment.py")

spec = importlib.util.spec_from_file_location("compressdiff", compressdiff_path)
compressdiff = importlib.util.module_from_spec(spec)
sys.modules["compressdiff"] = compressdiff
spec.loader.exec_module(compressdiff)

In [3]:
ROOT_DIR = "C:\\Adrianov\\Projects\\Project-Satanael\\"
DEMO_DIR = os.path.join(ROOT_DIR, 'attack_demo')
APRICOT_TEST_DIR = os.path.join(ROOT_DIR, 'data', 'APRICOTv1.0', 'Images', 'Test')
TJUDHD_TEST_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random', 'images')

In [4]:
def save_heatmap(img, path, cmap='grey'):
    plt.imsave(path, img, cmap=cmap)

In [5]:
print(filenames_combi)

[]


## Segment Patched Dataset with Compression Difference Only

In [ ]:
from skimage.morphology import disk
from skimage.filters import threshold_local

# radius = 15
# selem = disk(radius)
kernel_pram = 60

filenames_combi = []
DATA_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched')
i = len(filenames_combi)

savefig_path = os.path.join(ROOT_DIR, 'results_cd_grey_test_final_eval')
if not os.path.exists(savefig_path):
    os.makedirs(savefig_path)

dirs = ['Test', 'Train']

patch_types = [
    'Naturalistic1', 'Naturalistic2', 'Naturalistic3',
    'Naturalistic4', 'Naturalistic5', 'Naturalistic6',
    'TSEA1', 'TSEA2'
]

for d in dirs:
    for patch in patch_types:
        path = os.path.join(DATA_DIR, d, patch, 'images')
        for root, _, files in os.walk(path):
            for file in files:
                if (file in filenames_combi):
                    continue
                if not file.lower().endswith('.jpg'):
                    continue
                
                impath = os.path.join(path, file)
                dispImages = compressdiff.heatmap_recompressdiff(impath)
                average_OutputMap = np.mean(dispImages, axis=0)
                OutputMap_max = np.max(average_OutputMap)
                OutputMap_min = np.min(average_OutputMap)
                out_height = len(average_OutputMap)
                out_width = len(average_OutputMap[0])
                average_OutputMap = [int((average_OutputMap[i][j]-OutputMap_min)*255/(OutputMap_max-OutputMap_min)) for i in range(out_height) for j in range(out_width)]
                flatNumpyArray = np.array(average_OutputMap,dtype=np.uint8)
                
                # Convert the array to make a grayscale image
                grayImage = flatNumpyArray.reshape(out_height, out_width)
                img = cv2.imread(impath)
                ori_height, ori_width, _ = img.shape
                grayImage = cv2.resize(grayImage, (ori_width, ori_height)) 
    
                # Morphological processing
                base_kernel_size = int(min(ori_height, ori_width)/kernel_pram)
                kernel=np.ones((base_kernel_size*2,base_kernel_size*2),np.uint8)
                opened = cv2.morphologyEx(grayImage, cv2.MORPH_OPEN,kernel, iterations=1)
                kernel=np.ones((base_kernel_size,base_kernel_size),np.uint8)
                closed=cv2.morphologyEx(opened,cv2.MORPH_CLOSE,kernel, iterations=2)
                kernel=np.ones((base_kernel_size*2,base_kernel_size*2),np.uint8)
                opened2=cv2.morphologyEx(closed,cv2.MORPH_OPEN,kernel, iterations=2)
                non_zero_pixels = opened2[opened2 > 0]
                mean_intensity = np.mean(non_zero_pixels)
                 
                filled = opened2.copy()
                filled[filled < mean_intensity] = mean_intensity
                 
                filled = filled.astype(np.uint8)
                
                # Apply Otsu’s threshold
                _, thresh_map_adversarial = cv2.threshold(
                    filled, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
                )
                    
                save_heatmap(grayImage, os.path.join(savefig_path, file + "_cd.png"))
                save_heatmap(opened, os.path.join(savefig_path, file + "_cd_o.png"))
                save_heatmap(closed, os.path.join(savefig_path, file + "_cd_o_c.png"))
                save_heatmap(opened2, os.path.join(savefig_path, file + "_cd_o_c_o.png"))
                save_heatmap(filled, os.path.join(savefig_path, file + "_cd_filled.png"))
                # save_heatmap(thresh_map_local, os.path.join(savefig_path, file + "_cd_local.png"))
                save_heatmap(thresh_map_adversarial, os.path.join(savefig_path, file + "_cd_thresh.png"))
                i += 1
                print(f"File {file} saved. {i} heatmap(s) processed")
                filenames_combi.append(file)
                

height , width 1200 1624
File Naturalistic3_1499822101980.jpg saved. 1222 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_1499822871656.jpg saved. 1223 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_1499822894713.jpg saved. 1224 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_1499822940920.jpg saved. 1225 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_1499823079137.jpg saved. 1226 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_1499823704245.jpg saved. 1227 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_1499824076400.jpg saved. 1228 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_1499824241885.jpg saved. 1229 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_1499824590030.jpg saved. 1230 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_1499824844997.jpg saved. 1231 heatmap(s) processed
height , width 1200 1624
File Naturalistic3_149982

In [42]:
print(filenames_combi)
print(len(filenames_combi)) 

['Naturalistic1_1499917703635.jpg', 'Naturalistic1_1499918339461.jpg', 'Naturalistic1_1499923500376.jpg', 'Naturalistic1_1499925371739.jpg', 'Naturalistic1_1499931818829.jpg', 'Naturalistic1_1499932367119.jpg', 'Naturalistic1_1499933971246.jpg', 'Naturalistic1_1499934217508.jpg', 'Naturalistic1_1499934495500.jpg', 'Naturalistic1_1499939610334.jpg', 'Naturalistic1_1499940414097.jpg', 'Naturalistic1_1499940503860.jpg', 'Naturalistic1_1499940647224.jpg', 'Naturalistic1_1499991199874.jpg', 'Naturalistic1_1499995387796.jpg', 'Naturalistic1_1499997243748.jpg', 'Naturalistic1_1499997506250.jpg', 'Naturalistic1_1499997891867.jpg', 'Naturalistic1_1499998395606.jpg', 'Naturalistic1_1499998680556.jpg', 'Naturalistic1_1499998720024.jpg', 'Naturalistic1_1499999051151.jpg', 'Naturalistic1_1499999078373.jpg', 'Naturalistic1_1500000581873.jpg', 'Naturalistic1_1500001935862.jpg', 'Naturalistic1_1500002615319.jpg', 'Naturalistic1_1500004891513.jpg', 'Naturalistic1_1500004915796.jpg', 'Naturalistic1_1500

In [18]:
print(filenames_combi)
print(len(filenames_combi))

['1496728971354.jpg', '1496789537998.jpg', '1496793883119.jpg', '1496795772097.jpg', '1496803310299.jpg', '1496823270363.jpg', '1496830925210.jpg', '1496831236394.jpg', '1496841113093.jpg', '1496841138204.jpg', '1496841947868.jpg', '1496842005463.jpg', '1496846063139.jpg', '1496876160962.jpg', '1496876354624.jpg', '1496876707265.jpg', '1496876795911.jpg', '1496886272966.jpg', '1496886310687.jpg', '1496887051642.jpg', '1496887105399.jpg', '1496887944166.jpg', '1496888007783.jpg', '1496888095096.jpg', '1496888423196.jpg', '1496888877141.jpg', '1496889026902.jpg', '1496889070878.jpg', '1496890249554.jpg', '1496890894443.jpg', '1496891261216.jpg', '1496900532589.jpg', '1496911192932.jpg', '1496916190937.jpg', '1496916202450.jpg', '1496916450600.jpg', '1496917905645.jpg', '1496923078149.jpg', '1496927473512.jpg', '1496927956130.jpg', '1496927967487.jpg', '1496928151661.jpg', '1496928209210.jpg', '1496929694660.jpg', '1496991946111.jpg', '1496993189573.jpg', '1496993621491.jpg', '14970150536